# Sistem Clustering Kemiskinan Wilayah di Indonesia
## K-Means Clustering — Modeling

**Alur notebook:**
1. Import library
2. Load dataset
3. Eksplorasi awal (EDA)
4. Preprocessing: seleksi fitur, cek multikolinearitas (VIF), scaling
5. Elbow Method
6. Silhouette Score
7. Training model K-Means final
8. Evaluasi cluster (Silhouette, DBI, CHI)
9. Visualisasi
10. Simpan output

---
## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.preprocessing import RobustScaler

# Clustering
from sklearn.cluster import KMeans

# Evaluasi
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score

# Seleksi fitur
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
from statsmodels.tools.tools import add_constant

print('Library berhasil diimport.')

---
## 2. Load Dataset

In [ ]:
df = pd.read_excel('/content/Data_Tingkat_Kemiskinan.xlsx')

print('Shape data:', df.shape)
print()
df.head()

---
## 3. Eksplorasi Awal (EDA)

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
print('Missing value per kolom:')
print(df.isnull().sum())
print()
print('Duplikat:', df.duplicated().sum())

In [ ]:
# distribusi fitur target
plt.figure(figsize=(8, 4))
sns.histplot(df['Tingkat_Penduduk_Miskin'], bins=30, kde=True, color='steelblue')
plt.xlabel('Tingkat Penduduk Miskin (%)')
plt.title('Distribusi Tingkat Penduduk Miskin')
plt.tight_layout()
plt.show()

In [ ]:
# heatmap korelasi
plt.figure(figsize=(10, 7))
corr = df.drop(columns=['Kabupaten_Kota']).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', mask=mask,
            cmap='coolwarm', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Heatmap Korelasi Antar Fitur', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Preprocessing

### 4.1 Cleaning

In [ ]:
df = df.dropna().drop_duplicates().reset_index(drop=True)
print('Shape setelah cleaning:', df.shape)

### 4.2 Seleksi Fitur — VIF (Variance Inflation Factor)

VIF mengukur seberapa besar sebuah fitur bisa diprediksi dari fitur-fitur lain.

- **VIF < 5** → aman, tidak ada multikolinearitas
- **VIF 5–10** → perlu diwaspadai
- **VIF > 10** → fitur redundan, sebaiknya dihapus

Fitur yang sangat berkorelasi satu sama lain memberi informasi ganda ke K-Means dan merusak kualitas cluster.

In [ ]:
fitur_numerik = df.drop(columns=['Kabupaten_Kota', 'Tingkat_Penduduk_Miskin'])

X_vif = add_constant(fitur_numerik)
vif_df = pd.DataFrame({
    'Fitur': fitur_numerik.columns,
    'VIF Score': [vif(X_vif.values, i+1) for i in range(fitur_numerik.shape[1])]
}).sort_values('VIF Score', ascending=False).reset_index(drop=True)

vif_df

In [ ]:
# Drop Harapan Lama Sekolah — VIF tinggi & korelasi 0.78 dengan Rata-rata Lama Sekolah
# (informasi yang sama, pilih salah satu saja)
df = df.drop(columns=['Harapan Lama Sekolah'])

# Cek VIF ulang setelah drop
fitur_numerik = df.drop(columns=['Kabupaten_Kota', 'Tingkat_Penduduk_Miskin'])
X_vif2 = add_constant(fitur_numerik)
vif_df2 = pd.DataFrame({
    'Fitur': fitur_numerik.columns,
    'VIF Score': [vif(X_vif2.values, i+1) for i in range(fitur_numerik.shape[1])]
}).sort_values('VIF Score', ascending=False).reset_index(drop=True)

print('VIF setelah drop Harapan Lama Sekolah:')
vif_df2

### 4.3 Definisi Fitur untuk Clustering

Semua fitur numerik yang tersisa digunakan — VIF sudah aman semua (< 5).

In [ ]:
features = [
    'Tingkat_Penduduk_Miskin',
    'Rata-rata Lama Sekolah',
    'Indeks Pembangunan Gender',
    'Usia Harapan Hidup',
    'PengeluaranPerKapita',
    'Produk Domestik Regional Bruto',
    'Indeks Kemahalan Konstruksi',
    'PengeluaranPerkapita_Rokok'
]

X = df[features].copy()
print('Fitur yang digunakan:', features)
print('Shape X:', X.shape)

### 4.4 Scaling — RobustScaler

Dataset kemiskinan punya beberapa fitur yang distribusinya **sangat skewed** (miring), terutama `Produk Domestik Regional Bruto` yang nilainya bisa sangat ekstrem di kota besar.

**Kenapa RobustScaler, bukan MinMaxScaler?**

MinMaxScaler menarik semua nilai ke rentang 0–1 menggunakan min dan max. Tapi kalau ada 1 nilai ekstrem (outlier), seluruh data lain akan 'terjepit' di area sempit. RobustScaler menggunakan **median dan IQR** — tidak terpengaruh outlier.

**Analogi:** Bayangkan ngitung rata-rata gaji di satu kantor. Kalau ada 1 orang yang gajinya 100x lebih besar dari yang lain, rata-rata jadi tidak representatif. Tapi kalau pakai median, angkanya tetap wajar. Itulah cara kerja RobustScaler.

In [ ]:
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=features)
print('Data setelah RobustScaler:')
X_scaled_df.describe().round(3)

---
## 5. Elbow Method — Menentukan K Optimal

Plot **inertia** (WCSS) untuk K=1 hingga 10. K optimal ada di titik siku — penambahan K setelah titik itu tidak banyak menurunkan inertia.

In [ ]:
inertia_list = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=42)
    km.fit(X_scaled)
    inertia_list.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(list(K_range), inertia_list, marker='o', color='steelblue', linewidth=2, markersize=8)
plt.xlabel('Jumlah Cluster (K)', fontsize=12)
plt.ylabel('Inertia (WCSS)', fontsize=12)
plt.title('Elbow Method — Menentukan K Optimal', fontsize=14)
plt.xticks(list(K_range))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
## 6. Silhouette Score — Konfirmasi K

Silhouette Score mengukur seberapa baik tiap data 'cocok' di clusternya sendiri dibanding cluster lain.
Nilai mendekati **1.0 = cluster sangat bagus dan terpisah jelas**.

In [ ]:
sil_scores = {}

for k in range(2, 11):
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=42)
    lbl = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, lbl)
    sil_scores[k] = score
    print(f'K={k:2d}  →  Silhouette Score: {score:.4f}')

best_k = max(sil_scores, key=sil_scores.get)
print(f'\n>>> K terbaik: K = {best_k}  (score = {sil_scores[best_k]:.4f})')

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(list(sil_scores.keys()), list(sil_scores.values()),
         marker='s', color='coral', linewidth=2, markersize=8)
plt.axvline(x=best_k, color='gray', linestyle='--', alpha=0.7, label=f'K optimal = {best_k}')
plt.xlabel('Jumlah Cluster (K)', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Silhouette Score tiap K', fontsize=14)
plt.xticks(list(sil_scores.keys()))
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
## 7. Training Model K-Means Final

Menggunakan:
- `init='k-means++'` → inisialisasi centroid lebih cerdas, lebih cepat konvergen & hasil lebih stabil
- `n_init=20` → jalankan 20 kali dengan seed berbeda, ambil yang terbaik
- `max_iter=500` → beri ruang lebih untuk konvergensi

In [ ]:
K_OPTIMAL = best_k

kmeans_final = KMeans(
    n_clusters=K_OPTIMAL,
    init='k-means++',
    n_init=20,
    max_iter=500,
    random_state=42
)

df['Cluster'] = kmeans_final.fit_predict(X_scaled)

print(f'Model selesai ditraining dengan K = {K_OPTIMAL}')
print(f'Iterasi hingga konvergen: {kmeans_final.n_iter_}')
print(f'Inertia final           : {kmeans_final.inertia_:.4f}')
print(f'\nDistribusi cluster:')
print(df['Cluster'].value_counts().sort_index())

---
## 8. Evaluasi Model

Tiga metrik evaluasi clustering yang digunakan:

| Metrik | Bagus jika | Arti |
|---|---|---|
| Silhouette Score | Mendekati **1** | Cluster padat & terpisah jelas |
| Davies-Bouldin Index | Mendekati **0** | Cluster tidak tumpang tindih |
| Calinski-Harabasz Index | Semakin **besar** | Cluster sangat terpisah & kompak |

In [ ]:
labels_final = df['Cluster'].values

sil  = silhouette_score(X_scaled, labels_final)
dbi  = davies_bouldin_score(X_scaled, labels_final)
chi  = calinski_harabasz_score(X_scaled, labels_final)

print('='*45)
print(f'  Silhouette Score         : {sil:.4f}')
print(f'  Davies-Bouldin Index     : {dbi:.4f}  (lebih kecil = lebih baik)')
print(f'  Calinski-Harabasz Index  : {chi:.2f}  (lebih besar = lebih baik)')
print('='*45)

In [ ]:
# Silhouette plot per cluster
sil_vals = silhouette_samples(X_scaled, labels_final)

fig, ax = plt.subplots(figsize=(8, 5))
warna = cm.get_cmap('Set1', K_OPTIMAL)
y_lower = 10

for c in range(K_OPTIMAL):
    sil_c = np.sort(sil_vals[labels_final == c])
    size_c = sil_c.shape[0]
    y_upper = y_lower + size_c
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_c,
                     facecolor=warna(c), alpha=0.8, label=f'Cluster {c}')
    y_lower = y_upper + 10

ax.axvline(x=sil, color='red', linestyle='--', linewidth=1.5, label=f'Rata-rata = {sil:.3f}')
ax.set_xlabel('Silhouette Coefficient', fontsize=11)
ax.set_ylabel('Cluster', fontsize=11)
ax.set_title(f'Silhouette Plot per Cluster (K={K_OPTIMAL})', fontsize=13)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## 9. Interpretasi & Pelabelan Cluster

In [ ]:
# Rata-rata fitur per cluster
ringkasan = df.groupby('Cluster')[features].mean().round(2)
print('Rata-rata fitur per cluster:')
ringkasan

In [ ]:
# Beri label berdasarkan ranking Tingkat_Penduduk_Miskin
pilihan_label = {
    2: ['Tidak Miskin', 'Miskin'],
    3: ['Tidak Miskin', 'Miskin', 'Sangat Miskin'],
    4: ['Tidak Miskin', 'Cukup Miskin', 'Miskin', 'Sangat Miskin'],
    5: ['Tidak Miskin', 'Cukup Miskin', 'Miskin', 'Sangat Miskin', 'Ekstrem Miskin'],
}

daftar_label = pilihan_label.get(K_OPTIMAL, [f'Cluster {i}' for i in range(K_OPTIMAL)])
rank_miskin  = ringkasan['Tingkat_Penduduk_Miskin'].rank(ascending=True)
label_map    = {cid: daftar_label[int(r)-1] for cid, r in rank_miskin.items()}

df['Kategori'] = df['Cluster'].map(label_map)

print('Mapping label:')
for c, lbl in sorted(label_map.items()):
    n_wil = (df['Cluster'] == c).sum()
    rata_miskin = df[df['Cluster'] == c]['Tingkat_Penduduk_Miskin'].mean()
    print(f'  Cluster {c} → {lbl:20s} | {n_wil:3d} wilayah | rata-rata miskin: {rata_miskin:.2f}%')

In [ ]:
# Tabel ringkasan
ringkasan['Kategori']       = ringkasan.index.map(label_map)
ringkasan['Jumlah Wilayah'] = df.groupby('Cluster').size()
ringkasan[['Kategori', 'Jumlah Wilayah'] + features]

---
## 10. Visualisasi

In [ ]:
warna = cm.get_cmap('Set1', K_OPTIMAL)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Plot 1: Tingkat Miskin vs Pengeluaran ---
for c in range(K_OPTIMAL):
    mask = labels_final == c
    axes[0].scatter(X_scaled[mask, 0], X_scaled[mask, 4],
                    label=label_map.get(c, f'Cluster {c}'),
                    alpha=0.7, s=50, color=warna(c))

# centroid
axes[0].scatter(kmeans_final.cluster_centers_[:, 0],
                kmeans_final.cluster_centers_[:, 4],
                marker='X', s=250, color='black', zorder=5, label='Centroid')
axes[0].set_xlabel('Tingkat Penduduk Miskin (scaled)', fontsize=10)
axes[0].set_ylabel('Pengeluaran per Kapita (scaled)', fontsize=10)
axes[0].set_title('Kemiskinan vs Pengeluaran per Kapita', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, linestyle='--', alpha=0.4)

# --- Plot 2: Tingkat Miskin vs Rata-rata Lama Sekolah ---
for c in range(K_OPTIMAL):
    mask = labels_final == c
    axes[1].scatter(X_scaled[mask, 0], X_scaled[mask, 1],
                    label=label_map.get(c, f'Cluster {c}'),
                    alpha=0.7, s=50, color=warna(c))

axes[1].scatter(kmeans_final.cluster_centers_[:, 0],
                kmeans_final.cluster_centers_[:, 1],
                marker='X', s=250, color='black', zorder=5, label='Centroid')
axes[1].set_xlabel('Tingkat Penduduk Miskin (scaled)', fontsize=10)
axes[1].set_ylabel('Rata-rata Lama Sekolah (scaled)', fontsize=10)
axes[1].set_title('Kemiskinan vs Rata-rata Lama Sekolah', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, linestyle='--', alpha=0.4)

plt.suptitle(f'Hasil K-Means Clustering — K={K_OPTIMAL}', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart distribusi wilayah per cluster
urutan = [label_map[c] for c in sorted(label_map.keys())]
distribusi = df['Kategori'].value_counts().reindex(urutan)

plt.figure(figsize=(9, 5))
bars = plt.bar(distribusi.index, distribusi.values,
               color=[warna(c) for c in range(K_OPTIMAL)],
               edgecolor='black', width=0.5)

for bar, val in zip(bars, distribusi.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.5,
             str(val), ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.xlabel('Kategori Kemiskinan', fontsize=12)
plt.ylabel('Jumlah Wilayah (Kab/Kota)', fontsize=12)
plt.title('Distribusi Wilayah per Cluster Kemiskinan', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot fitur utama per cluster
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

fitur_box = [
    ('Tingkat_Penduduk_Miskin', 'Tingkat Penduduk Miskin (%)'),
    ('PengeluaranPerKapita',    'Pengeluaran per Kapita'),
    ('Rata-rata Lama Sekolah',  'Rata-rata Lama Sekolah (Tahun)'),
]

for ax, (col, label) in zip(axes, fitur_box):
    data_box = [df[df['Cluster'] == c][col].values for c in sorted(df['Cluster'].unique())]
    bp = ax.boxplot(data_box, patch_artist=True, notch=False)
    for patch, c in zip(bp['boxes'], range(K_OPTIMAL)):
        patch.set_facecolor(warna(c))
        patch.set_alpha(0.7)
    ax.set_xticklabels([label_map.get(c, f'C{c}') for c in sorted(df['Cluster'].unique())],
                       fontsize=9, rotation=15)
    ax.set_title(label, fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Distribusi Fitur per Cluster', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 11. Sampel Wilayah per Cluster

In [ ]:
kolom_tampil = ['Kabupaten_Kota', 'Kategori',
                'Tingkat_Penduduk_Miskin', 'PengeluaranPerKapita', 'Rata-rata Lama Sekolah']

for c in sorted(df['Cluster'].unique()):
    lbl    = label_map.get(c, f'Cluster {c}')
    subset = df[df['Cluster'] == c][kolom_tampil]
    print(f'\n{"="*65}')
    print(f'  CLUSTER {c} — {lbl}  ({len(subset)} wilayah)')
    print(f'{"="*65}')
    print(subset.head(10).to_string(index=False))

---
## 12. Simpan Output untuk Tim Geospatial

In [ ]:
df_output = df[['Kabupaten_Kota', 'Cluster', 'Kategori'] + features].copy()
df_output.to_csv('hasil_clustering_kemiskinan.csv', index=False)

print('File tersimpan: hasil_clustering_kemiskinan.csv')
print(f'Total baris  : {len(df_output)}')
print(f'Kolom        : {df_output.columns.tolist()}')
print()
print(f'Ringkasan evaluasi model:')
print(f'  K optimal            : {K_OPTIMAL}')
print(f'  Silhouette Score     : {sil:.4f}')
print(f'  Davies-Bouldin Index : {dbi:.4f}')
print(f'  Calinski-Harabasz    : {chi:.2f}')
print()
df_output.head(10)